## Part 1: Parsing a Sentence with CKY

Given the following grammar in CNF.
1. Write the code for CKY algorithm and verify if the **sentence** follows or can be parsed by this grammar or not.
2. Also print the final CKY chart/table.

In [ ]:
import nltk
from nltk.parse import ChartParser
from nltk.grammar import CFG

In [ ]:
cky_grammar = CFG.fromstring("""
    S -> NP VP
    NP -> Det N
    VP -> V NP
    Det -> 'the'
    N -> 'man' | 'dog'
    V -> 'saw'
""")

sentence = "the man saw the dog"

In [ ]:
# Extract binary and lexical rules from cky_grammar (inspired by Demo)
grammar_dict = {
    "binary": [],
    "lexical": {}
}

for prod in cky_grammar.productions():
    lhs = prod.lhs().symbol()
    rhs = prod.rhs()
    if len(rhs) == 1 and isinstance(rhs[0], str):
        word = rhs[0]
        grammar_dict["lexical"].setdefault(word, []).append(lhs)
    elif len(rhs) == 2:
        b = rhs[0].symbol() if hasattr(rhs[0], "symbol") else str(rhs[0])
        c = rhs[1].symbol() if hasattr(rhs[1], "symbol") else str(rhs[1])
        grammar_dict["binary"].append((lhs, b, c))

def cky_parse(words, grammar, start="S"):
    n = len(words)
    # Creates the CKY chart, where each cell stores the grammar categories
    chart = [[set() for _ in range(n + 1)] for _ in range(n)]

    # Step 1: Lexical categories for individual words (length = 1)
    for i, word in enumerate(words):
        chart[i][i+1].update(grammar["lexical"].get(word, []))

    # Step 2: Build larger phrases (length 2 to n)
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            # Try every possible split point k
            for k in range(i + 1, j):
                # Apply binary grammar rules
                for lhs, b, c in grammar["binary"]:
                    if b in chart[i][k] and c in chart[k][j]:
                        chart[i][j].add(lhs)

    can_parse = start in chart[0][n]
    return chart, can_parse

def print_cky_chart(chart, words):
    n = len(words)
    print("=" * 60)
    print("                     CKY CHART TABLE")
    print("=" * 60)
    print("Span [i:j]   | Span Text                 | Categories")
    print("-" * 60)
    for length in range(1, n + 1):
        for i in range(n - length + 1):
            j = i + length
            span_text = " ".join(words[i:j])
            cats = str(chart[i][j]) if chart[i][j] else "set()"
            print(f"chart[{i}][{j}] | {span_text:<25} | {cats}")
    print("=" * 60)

# Run CKY parser
words = sentence.split()
chart, result = cky_parse(words, grammar_dict, start=cky_grammar.start().symbol())

print("Sentence:", sentence)
print("Can be parsed?", result)
print()
print_cky_chart(chart, words)


## Part 2: Comparision with NLTK's chart parser
After implementing your CKY algorithm, compare its output for the given sentence with the output of NLTK's built-in chart parser using the `cky_grammar`. This comparison will help you verify if your implementation is correct.

In [ ]:
# Compare with NLTK ChartParser (as in Demo Section 3)
parser = ChartParser(cky_grammar)
trees = list(parser.parse(words))

print("Number of parse trees from NLTK:", len(trees))
for i, tree in enumerate(trees, 1):
    print("")
    print(f"Parse Tree {i}:")
    print(tree)
    tree.pretty_print()

print("Comparison Result:")
print("Custom CKY parsed?:", result)
print("NLTK parsed?:", len(trees) > 0)
print("Do they match?:", result == (len(trees) > 0))
